# PyTorch Datasets and DataLoaders
This demo covers PyTorch Datasets and DataLoaders. We will pickup and cover the topics from the video. 

# PyTorch Datasets

### PyTorch Pre-loaded Datasets
Let's begin by covering "pre-loaded" Datasets in PyTorch

These are perfect for beginning working with Datasets or for research/experimentation.

Pre-loaded Datasets available: Image, Text and Audio

In [1]:
# Let's begin with Pre-loaded Audio files
# Import torchaudio 
import torchaudio.datasets

# To get a list of available Audio Datasets go to Documentation URL: https://pytorch.org/audio/stable/datasets.html

In [ ]:
# Create a dataset using DR_VCTK (Device Recorded VCTK https://pytorch.org/audio/stable/references.html#id42)
audio_dataset = torchaudio.datasets.DR_VCTK(root='./audio', subset='test', download=True) 


In [4]:
# Lets do a pre-loaded image dataset
# Import the torchvision datasets library
import torchvision.datasets
from torchvision.transforms import ToTensor

In [5]:
# Create the MNIST dataset used in the lab
mnist_dataset = torchvision.datasets.MNIST(
    root='./mnist',
    train=True,
    download=True,
    transform=ToTensor(),
)
print(f"MNIST training samples: {len(mnist_dataset)}")

# FashionMNIST uses the same Dataset API and gives us descriptive class names
image_dataset = torchvision.datasets.FashionMNIST(root='./fashion', train=False, download=True, transform=ToTensor())

# ToTensor converts each image to the tensor format expected by the model.

In [ ]:
# Let's print the Classes of a dataset 
print(image_dataset.classes)

In [ ]:
# Print the classes to their indexes 
print(image_dataset.class_to_idx)

# This is an attribute used to map class names to integer values because models require integer values for training


In [ ]:
# Reverse the class to index mapping for plotting
class_to_index_map = image_dataset.class_to_idx
index_to_class_map = {v: k for k, v in class_to_index_map.items()}
print(index_to_class_map)

In [ ]:
# Lets get a visual of our dataset with 9 random images
import torch
import matplotlib.pyplot as plt


# Set up our plot
figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(image_dataset), size=(1,)).item()
    img, label = image_dataset[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(index_to_class_map[label])
    plt.axis("off")
    plt.imshow(img.squeeze())
plt.show()

# PyTorch DataLoaders
Now that we have a working dataset, lets begin defining how we are going to present or load our data to our model.

This is done using DataLoaders!

In [12]:
# Import DataLoader
from torch.utils.data import DataLoader

In [14]:
# Create a new dataloader from our image_dataset above
image_dataloader = DataLoader(dataset=image_dataset, batch_size=64, shuffle=True, num_workers=1)

### DataLoader Parameters review
batch_size: Number of samples (images) are loaded at a time.

shuffle: When True, images are randomized before sending to the model.

num_workers: Number of processes to use for loading data. 

## Configuring a DataLoader

A `Dataset` owns samples and labels. A `DataLoader` controls how those samples are iterated, batched, shuffled, and loaded. The following in-memory dataset lets us focus on loader behavior without downloading additional data.


In [ ]:
from torch.utils.data import TensorDataset

features = torch.arange(23 * 4, dtype=torch.float32).reshape(23, 4)
labels = torch.arange(23) % 2
configuration_dataset = TensorDataset(features, labels)

print(f"Dataset samples: {len(configuration_dataset)}")


### Batching, Shuffling, Workers, and Pinned Memory

- `batch_size` controls the number of samples per batch.
- `shuffle=True` randomizes sample order for training.
- `num_workers` controls data-loading subprocesses; start with 0 in notebooks.
- `pin_memory=True` can accelerate CPU-to-CUDA transfers.
- `drop_last=True` discards an incomplete final batch.


In [ ]:
seeded_generator = torch.Generator().manual_seed(42)
configured_loader = DataLoader(
    configuration_dataset,
    batch_size=6,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
    generator=seeded_generator,
)


### Inspecting a Batch

The leading tensor dimension normally equals `batch_size`. The final batch may be smaller when the dataset size is not evenly divisible by the batch size.


In [ ]:
batch_features, batch_labels = next(iter(configured_loader))

print(f"Number of batches: {len(configured_loader)}")
print(f"Feature batch shape: {batch_features.shape}")
print(f"Label batch shape: {batch_labels.shape}")


### Relating Dataset Size, Batch Size, and Batch Shapes

When the dataset size is evenly divisible by `batch_size`, the number of batches is `len(dataset) // batch_size`. The first dimension of each returned tensor is the batch size; the remaining dimensions describe one sample. You will apply this same pattern with different values in the lab.


In [ ]:
example_features = torch.rand(24, 3, 16, 16)
example_labels = torch.randint(0, 2, (24,))
batching_dataset = TensorDataset(example_features, example_labels)

example_batch_size = 6
batching_loader = DataLoader(
    batching_dataset, batch_size=example_batch_size, shuffle=False
)
feature_batch, label_batch = next(iter(batching_loader))

print(f"Dataset samples: {len(batching_dataset)}")
print(f"Batch size: {example_batch_size}")
print(f"Number of batches: {len(batching_loader)}")
print(f"Feature batch shape: {feature_batch.shape}")
print(f"Label batch shape: {label_batch.shape}")


### Comparing `drop_last`

Dropping the final incomplete batch can be useful when every training batch must have the same size. Validation and testing normally keep every sample.


In [ ]:
keep_last_loader = DataLoader(configuration_dataset, batch_size=6, drop_last=False)
drop_last_loader = DataLoader(configuration_dataset, batch_size=6, drop_last=True)

print(f"Batches with drop_last=False: {len(keep_last_loader)}")
print(f"Batches with drop_last=True: {len(drop_last_loader)}")


### Reproducible Shuffling

Pass a separately seeded `torch.Generator` whenever shuffled order needs to be repeatable. Reusing the same generator object advances its state, so create a freshly seeded generator for each repeat.


In [ ]:
def first_seeded_batch(seed):
    loader = DataLoader(
        configuration_dataset,
        batch_size=6,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    return next(iter(loader))[0]

first_batch = first_seeded_batch(42)
repeated_batch = first_seeded_batch(42)
print(f"Same seed produces the same batch: {torch.equal(first_batch, repeated_batch)}")


### Training and Validation Shuffle Behavior

Training data is normally shuffled each epoch so the model does not repeatedly see samples in the same order. Validation data normally uses `shuffle=False` to preserve deterministic ordering.


In [ ]:
training_loader = DataLoader(
    configuration_dataset, batch_size=4, shuffle=True
)
validation_loader = DataLoader(
    configuration_dataset, batch_size=4, shuffle=False
)

print(f"Training sampler: {type(training_loader.sampler).__name__}")
print(f"Validation sampler: {type(validation_loader.sampler).__name__}")


### Performance Considerations

Increase `num_workers` only after measuring an input-pipeline bottleneck. More workers add process and memory overhead. Pinned memory is mainly useful for CUDA transfers and is not a universal speed improvement.


In [ ]:
# iterate through the image_dataloader
features, labels = next(iter(image_dataloader))
# Print the batch size and the number of labels
print(f"Features shape: {features.size()}")
print(f"Labels shape: {labels.size()}")

In [ ]:
# Show an image and label from a random position in the batch.
import random

random_index = random.randrange(labels.size(0))
image = features[random_index].squeeze()
label = labels[random_index]

plt.imshow(image)
plt.axis("off")
plt.show()
print(f"Label: {label} -> {index_to_class_map[label.item()]}")


# Review
So far we have created Datasets using PyTorch's pre-loaded datasets.

We have also created a DataLoader used to present our dataset to our model.

# Custom Datasets
Lets take a look at how to create custom datasets in Pytorch using our own existing images.

In [19]:
# Import Dataset
from torch.utils.data import Dataset

In [20]:
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset


class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, class_list):
        self.df = pd.read_csv(annotations_file)
        self.class_list = class_list
        self.class_to_idx = {name: index for index, name in enumerate(class_list)}
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        image_path = self.df.file_path[index]
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)
        label = self.class_to_idx[self.df.label[index]]
        return image, label, image_path


In [ ]:
"""
Dataset Class Review

__init__ method: Peforms initial setup and load the data. 

__len__ method: Returns the number of samples for batch.

__getitem__ method: Retrieves a single data sample based on an index.

"""

In [22]:
# Lets create our custom dataset!
# We must define an annotations file and a list of classes
class_list = ["cat", "dog"]

In [ ]:
# Create custom dataset
custom_dataset = CustomImageDataset(annotations_file='labels.csv', class_list=class_list)
print(custom_dataset)

In [ ]:
# Print attributes of our dataset (__init__ method)

# Display our annotations
print(f"Annotations data: \n{custom_dataset.df}") 

In [ ]:
# Show our classes
print(f"Classes: {custom_dataset.class_list}")

In [ ]:
# Show our class to index map
print(f"Mapped Classes: {custom_dataset.class_to_idx}")

In [28]:
# Create our own mapper OR we could add this in our __getitem__ method
custom_class_labels_map = {0: 'cat', 1: 'dog'}

In [ ]:
# Lets get a visual of our dataset with 9 random images
import torch
import matplotlib.pyplot as plt
from PIL import Image


# Set up our plot
figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(custom_dataset), size=(1,)).item()
    img, label = custom_dataset[sample_idx][2], custom_dataset[sample_idx][1]
    img = Image.open(img)
    figure.add_subplot(rows, cols, i)
    plt.title(label)
    plt.axis("off")
    plt.imshow(img)
plt.show()

In [31]:
# Create a DataLoader for our custom dataset
custom_dataloader = DataLoader(dataset=custom_dataset, batch_size=64, shuffle=True)

In [ ]:
# Iterate through this dataloader like we did above
features, labels, urls = next(iter(custom_dataloader))
# Print the batch size and the number of labels
print(f"Features shape: {features.size()}")
print(f"Labels shape: {labels.size()}")

In [ ]:
# Show an image and label from a random position in the custom batch.
import random

random_index = random.randrange(labels.size(0))
image_path = urls[random_index]
label = labels[random_index]

image = Image.open(image_path)
plt.imshow(image)
plt.axis("off")
plt.show()
print(f"Label: {label} -> {custom_class_labels_map[label.item()]}")


# Torchvision ImageFolder
Create a dataset using the folder structure as a way to label your images.

This utility simplifies the process of loading datasets where images are organized in a directory structure.

Example:
```bash
images/
    ├── cat/
    │   ├── cat1.jpg
    │   ├── cat2.jpg
    ├── dog/
    │   ├── dog1.jpg
    │   └── dog2.jpg
```

Each image will be labeled by its directory. Before a `DataLoader` combines images into a batch, every image must have a compatible shape. A `Resize` transform makes the spatial dimensions consistent, and `ToTensor` converts each image to channel-first tensor format.


In [34]:
# Import torchvision
import torchvision

In [ ]:
# Resize images to a common shape before DataLoader collation
image_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

# Create a dataset using ImageFolder
image_folder_dataset = torchvision.datasets.ImageFolder(
    root="images",
    transform=image_transform,
)
print(image_folder_dataset)

In [ ]:
# Let's print the Classes of a dataset 
print(image_folder_dataset.classes)

In [ ]:
# Print the classes to their indexes 
print(image_folder_dataset.class_to_idx)

In [38]:
# Load into dataloader
image_folder_dataloader = DataLoader(image_folder_dataset, batch_size=64, shuffle=True)

In [39]:
# Retrieve one batch of images and labels
images, labels = next(iter(image_folder_dataloader))


In [ ]:
# Plot the batch above
fig, axes = plt.subplots(1, len(images), figsize=(8, 8))

for i, (img, label) in enumerate(zip(images, labels)):
    img = img.permute(1, 2, 0) # Ignore this
    axes[i].imshow(img)
    axes[i].set_title(image_folder_dataset.classes[label])
    axes[i].axis("off")
plt.show()